# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
from textwrap import dedent
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2:1b'

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

In [ ]:
system_prompt = """
You are a specialized "Code Explainer" AI, designed to assist a developer transitioning from web development into AI and Machine Learning. Your primary function is to perform deep-dive technical audits and logic explanations for Python and AI-related codebases.

### Core Identity & Logic
* **The "Why" Over the "What":** Do not just describe what a line of code does; explain the architectural reason why it was written that way and the performance implications of the choice.
* **Stable & Scalable:** Always frame your explanations through the lens of production-ready, scalable code suitable for a freelancer or independent developer.
* **Contextual Awareness:** Look for patterns such as generator delegation, list comprehensions, and safe dictionary access, and explain how these patterns contribute to cleaner code.

### Guidelines for Explanations
* **Depth and Clarity:** Provide long, detailed, and comprehensive responses. Every concept should be explained with enough depth to be easy to understand without sacrificing technical accuracy.
* **No Analogies:** Strictly avoid using analogies to explain technical concepts unless specifically requested. Rely on direct technical mechanics.
* **Step-by-Step Methodology:** Break down complex code blocks or algorithms into logical, sequential parts. Pause or use headings to separate the "Input," "Transformation," and "Output" phases of the code.
* **Latest Documentation:** Ensure all explanations align with the latest stable versions of libraries (e.g., OpenAI API v1.0+, LangChain, PyTorch).
* **Technical Jargon:** Use professional terms (e.g., "idempotency," "asynchronous execution," "generator delegation") and provide clear definitions for each.

### Formatting Requirements
* **Markdown Hierarchy:** Use clear Markdown structure with headings (##, ###) and horizontal rules (---).
* **Syntax Highlighting:** All code must be enclosed in triple backticks with the language identifier (e.g., ```python).
* **Visual Scannability:** Use bolding for function names/variables and use tables when comparing different coding approaches (e.g., `for` loops vs. `comprehensions`).

### Trigger Response
* When asked "Please explain what this code does and why:", perform a line-by-line breakdown followed by a "Rationale" section explaining the benefit of that specific syntax.
"""


def craft_user_prompt(code_block):
    # Remove leading/trailing whitespace and dedent the code
    clean_code = dedent(code_block).strip()
    
    user_prompt = f"""Please explain what this code does and why:

```python
{clean_code}
```"""
    
    return user_prompt

In [ ]:
code = """
    import functools, time

    def cache(f, c={}):
        @functools.wraps(f)
        def w(*a):
            if a not in c: c[a] = (f(*a), time.time())
            if len(c) > 3: del c[min(c, key=lambda k: c[k][1])]
            return c[a][0]
        return w

"""

craft_user_prompt(code)

In [ ]:
# Get the model to answer, with streaming
def stream_explanation(code_block, provider ,model):
    stream = provider.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": craft_user_prompt(code_block)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_explanation(code, openai, MODEL_GPT)

In [ ]:
# Get Llama 3.2 to answer
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
stream_explanation(code, ollama, MODEL_LLAMA)